In [125]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

print("All libraries imported successfully")

All libraries imported successfully


In [126]:

df = pd.read_csv("../data/cleaned/clean_startups.csv")

print("Dataset was loaded")

print("\nDataset Shape:", df.shape)

df.head()

Dataset was loaded

Dataset Shape: (51104, 10)


,industry,total_funding,status,country,region,city,funding_rounds,founded_date,first_funding_date,last_funding_date
0,Application Platforms|Real Time|Social Network...,700000.0,operating,USA,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
1,Curated Web,2000000.0,operating,CHN,Beijing,Beijing,1,2007-01-01,2008-03-19,2008-03-19
2,Software,2000000.0,operating,USA,"Springfield, Illinois",Champaign,1,2010-01-01,2014-07-24,2014-07-24
3,Biotechnology,762851.0,operating,CAN,Vancouver,Vancouver,2,1997-01-01,2009-09-11,2009-12-21
4,Analytics,33600000.0,operating,USA,SF Bay Area,Mountain View,4,2011-01-01,2013-01-03,2015-11-09


In [127]:
# Inspect the Dataset

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (51104, 10)

Columns:
Index(['industry', 'total_funding', 'status', 'country', 'region', 'city',
       'funding_rounds', 'founded_date', 'first_funding_date',
       'last_funding_date'],
      dtype='object')

Data Types:
industry               object
total_funding         float64
status                 object
country                object
region                 object
city                   object
funding_rounds          int64
founded_date           object
first_funding_date     object
last_funding_date      object
dtype: object

Missing Values:
industry              0
total_funding         0
status                0
country               0
region                0
city                  0
funding_rounds        0
founded_date          0
first_funding_date    0
last_funding_date     0
dtype: int64


In [128]:
date_columns = ["founded_date", "first_funding_date", "last_funding_date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print("Date columns converted")

print(df[date_columns].dtypes)

Date columns converted
founded_date          datetime64[ns]
first_funding_date    datetime64[ns]
last_funding_date     datetime64[ns]
dtype: object


In [129]:
current_year = pd.Timestamp.now().year

df["company_age"] = current_year - df["founded_date"].dt.year

print("Company Age feature created")

df[["founded_date", "company_age"]].head()

Company Age feature created


,founded_date,company_age
0,2014-09-04,12.0
1,2007-01-01,19.0
2,2010-01-01,16.0
3,1997-01-01,29.0
4,2011-01-01,15.0


In [130]:
df.drop(
    columns=[
        "founded_date",
        "first_funding_date",
        "last_funding_date"
    ],
    inplace=True
)

print("Date columns dropped")

df.head()

Date columns dropped


,industry,total_funding,status,country,region,city,funding_rounds,company_age
0,Application Platforms|Real Time|Social Network...,700000.0,operating,USA,DE - Other,Delaware City,2,12.0
1,Curated Web,2000000.0,operating,CHN,Beijing,Beijing,1,19.0
2,Software,2000000.0,operating,USA,"Springfield, Illinois",Champaign,1,16.0
3,Biotechnology,762851.0,operating,CAN,Vancouver,Vancouver,2,29.0
4,Analytics,33600000.0,operating,USA,SF Bay Area,Mountain View,4,15.0


In [131]:
categorical_columns = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:")
print(categorical_columns)

Categorical Columns:
Index(['industry', 'status', 'country', 'region', 'city'], dtype='object')


In [132]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in categorical_columns:
    encoder = LabelEncoder()

    df[col] = encoder.fit_transform(df[col])

    encoders[col] = encoder

print("Categorical columns encoded")

df.head()

Categorical columns encoded


,industry,total_funding,status,country,region,city,funding_rounds,company_age
0,4212,700000.0,3,122,226,976,2,12.0
1,12732,2000000.0,3,21,79,272,1,19.0
2,23072,2000000.0,3,122,858,677,1,16.0
3,7337,762851.0,3,18,963,4006,2,29.0
4,2352,33600000.0,3,122,788,2546,4,15.0


In [133]:
import joblib
import os

# Create models folder if it doesn't exist
os.makedirs("../models", exist_ok=True)

joblib.dump(encoders, "../models/encoder.pkl")

print("Encoders saved")

Encoders saved


In [134]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numerical_columns = [
    "total_funding",
    "funding_rounds",
    "company_age"
]

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

print("Numerical columns scaled")

df.head()

Numerical columns scaled


,industry,total_funding,status,country,region,city,funding_rounds,company_age
0,4212,-0.083409,3,122,226,976,0.105397,-0.605725
1,12732,-0.076459,3,21,79,272,-0.581880,0.073546
2,23072,-0.076459,3,122,858,677,-0.581880,-0.217570
3,7337,-0.083073,3,18,963,4006,0.105397,1.043934
4,2352,0.092475,3,122,788,2546,1.479950,-0.314609


In [135]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

print("Scaler saved")

Scaler saved


In [136]:
X = df.drop("status", axis=1)
y = df["status"]

print("Features and target created")

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features and target created
Features Shape: (51104, 7)
Target Shape: (51104,)


In [137]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train-Test split completed")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Train-Test split completed
X_train: (40883, 7)
X_test : (10221, 7)
y_train: (40883,)
y_test : (10221,)


In [138]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/processed.csv", index=False)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("All processed files saved successfully")

All processed files saved successfully
